In [5]:
import torch
import sys
import time

sys.path.append("/home/atuin/v120bb/v120bb18/UnReflectAnything")
from utilities.visualization import rgb, panelize
from polar_highlighter import PolarHighlighter

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    curr_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(curr_device)
    print(f"CUDA is available: {num_devices} device(s) detected.")
    print(f"Current device id: {curr_device} - {device_name}")
else:
    print("CUDA is not available")
%load_ext autoreload
%autoreload 2


CUDA is available: 1 device(s) detected.
Current device id: 0 - NVIDIA A40
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
from main import load_and_process_config
from models_utils import load_best_model_by_run
from dataset import from_config
from utilities import tensor_dict_summarize


config = load_and_process_config("config_train.yaml")
config.RUN = "curious-moon-751"
config.DATASETS = {"PSD": config.DATASETS.PSD}
dataset = from_config(config)["validation"]
idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True))
model = load_best_model_by_run(config.RUN).eval()

DATASET  [00:58:36] Processing 1 datasets: ['PSD']

DATASET  [00:58:36] Using dataset-specific TRAIN_SCENES for PSD: PSD_Train

DATASET  [00:58:36]   ✓ Created training dataset for PSD: 361 samples from specific scenes

DATASET  [00:58:36]   ✓ Created validation dataset for PSD: 471 samples from 7 scenes

DATASET  [00:58:36] === Dataset Creation Summary ===

...

MODEL    [00:58:39] Decoder 'diffuse' un-frozen due to DECODER_LR=0.0001

MODEL    [00:58:39] Decoder 'highlight' un-frozen due to DECODER_LR=0.0005

MODEL    [00:58:39] ✓ Token Inpainter: Successfully loaded all 78 state dict keys from weights/token_inpainter.pth

MODEL    [00:58:39] Loaded pretrained token inpainter weights from weights/token_inpainter.pth

MODEL    [00:58:39] Model with class UnReflect_Model_TokenInpainter created with 498,003,972 parameters

In [46]:
# torch.save(model.token_inpaint.state_dict(), "/home/atuin/v120bb/v120bb18/UnReflectAnything/weights/token_inpainter.pth")

In [47]:
batch.keys()

dict_keys(['raw', 'specular', 'diffuse', 'intrinsics'])

In [ ]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

for b, batch in enumerate(dataloader):
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
        for k, v in batch.items()
    }
    patch_mask = batch["raw"].mean(dim=1, keepdim=True) > 0.7
    model_input = {"rgb": batch["raw"], "inpaint_mask_override": patch_mask}
    with torch.no_grad():
        modelout = model(model_input)
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(512, 512),
        blackout=True,
        return_pca=True,
    )
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(512, 512)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(512, 512),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(512, 512),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(512, 512)),
        )
    )
    if b > 20:
        break
